# Experiment 7.1 — Long-tau anti-persistence regularization

This notebook is **analysis-only**. It compares aggregate Exp7.1 conditions and intentionally does not display individual training runs, per-seed rasters, histories, or evaluations.

Primary question: can long-layer-specific rate + anti-persistence regularization suppress sustained firing in the serial `s4 -> s5 -> s6` end-to-end SNN, and does full-trajectory regularization outperform valid-only regularization?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_1_long_tau_antipersistence' / 'long_tau_antipersistence_v1'

summary = pd.read_csv(ART / 'summary.csv')
dynamics = pd.read_csv(ART / 'dynamics_summary.csv')
paired = pd.read_csv(ART / 'paired_condition_deltas.csv')
calibration = pd.read_csv(ART / 'calibration_summary.csv')

order = ['no_reg', 'all_loss_masked', 'wholecount_only_masked']


## Classification comparison

Only condition-level mean ± SD is shown.

In [ ]:
display(summary.set_index('condition').reindex(order).reset_index())
frame = summary.set_index('condition').reindex(order)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(len(frame)), frame['test_ba_mean'], yerr=frame['test_ba_std'].fillna(0), capsize=4)
ax.set_xticks(range(len(frame)))
ax.set_xticklabels(frame.index, rotation=20, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_ylim(0, 1)
ax.set_title('Exp7.1 test BA by condition')
plt.tight_layout()
plt.show()


## Long-layer firing dynamics

The key success criterion is not merely lower firing. A successful condition should shorten persistent runs and reduce full/padding firing while keeping long-layer activity non-zero and preserving classification.

In [ ]:
cols = [
    'condition',
    'long_valid_firing_fraction_mean',
    'long_full_firing_fraction_mean',
    'long_padding_firing_fraction_mean',
    'long_mean_max_run_per_neuron_full_mean',
    'long_p95_max_run_per_neuron_full_mean',
    'long_current_rms_full_mean',
    'long_weight_fro_mean',
]
view = dynamics.set_index('condition').reindex(order).reset_index()
display(view[cols])

fig, ax = plt.subplots(figsize=(9, 4.8))
x = range(len(view))
w = 0.25
ax.bar([i-w for i in x], view['long_valid_firing_fraction_mean'], width=w, label='valid')
ax.bar(list(x), view['long_full_firing_fraction_mean'], width=w, label='full')
ax.bar([i+w for i in x], view['long_padding_firing_fraction_mean'], width=w, label='padding')
ax.set_xticks(list(x))
ax.set_xticklabels(view['condition'], rotation=20, ha='right')
ax.set_ylabel('Long-layer firing fraction')
ax.set_title('Long-layer firing occupancy')
ax.legend()
plt.tight_layout()
plt.show()


## Paired condition effects

The main contrast is `wholecount_only_masked - all_loss_masked`, which isolates the effect of regularizing the complete padded trajectory rather than only the valid interval. Results below are aggregated across the paired seeds.

In [ ]:
delta_cols = [c for c in paired.columns if c.endswith('_delta')]
paired_summary = paired.groupby('contrast')[delta_cols].agg(['mean', 'std'])
display(paired_summary)


## Gradient calibration

Regularized conditions are calibrated independently so that valid-only and full-trajectory policies have comparable initial regularizer-to-task gradient strength on `W_L`.

In [ ]:
display(calibration.set_index('condition').reindex(order).reset_index())


## Interpretation checklist

A regularized condition should only be called successful if: (1) continuous-run metrics decrease, (2) full/padding firing decreases, (3) long-layer firing remains non-zero, (4) long-current magnitude moves to a healthier regime, and (5) test BA is not materially degraded. If long activity collapses toward zero while BA survives, interpret that as pruning the long branch rather than repairing long-timescale memory.